In [ ]:
"""
Step 3d - POC Cohort Selection & Cohort Dataset (features -> POC cohort)
Customer Health & Sentiment Scoring Multi-Agent AI Solution

Input : data/features/customer_month_features.parquet   (Step 3c, FULL population)
        data/features/customer_month_labels.parquet     (Step 3c, evaluation only)
        data/silver/customers.parquet                    (static cohort attributes)
        config/cohort_selection.json                     (OPTIONAL business override, see below)
Output: data/cohort/poc_customer_month.parquet          model-ready POC rows (+ split column)
        data/cohort/poc_customer_month_labels.parquet   ground truth for the same rows (evaluation only)
        data/cohort/poc_customers.parquet               one row per cohort customer (+ split)
        reports/cohort/cohort_core_overall.csv          size of the core population under the rule used
        reports/cohort/cohort_rule_sizes.csv            core size under every CORE_RULE (same table as EDA 12A)
        reports/cohort/cohort_scorecard.csv             all candidates, ranked
        reports/cohort/cohort_validation.csv            gate checks
        reports/cohort/cohort_representativeness.csv    cohort vs full core population
        reports/cohort/cohort_coverage_by_month.csv
        reports/cohort/cohort_selection_used.json       what was selected and why

POC inclusion criteria (customer-month level) - identical to EDA Steps 10 / 12 (business_rules.py)
  C1  cleaned Silver customer that passed Step 3b and has >= 1 event (Step 3c spine)
  C2  active in the month (active_m = 1) AND months_observed >= MIN_HISTORY_MONTHS   (EDA HISTORY_COND)
  C3  same-month sources satisfy CORE_RULE                                             (EDA CORE_COND)
        tx_plus_one : has_txn_m AND (has_eng_m OR has_int_m)      <- EDA configured default
        any_two     : any 2 of has_txn_m / has_eng_m / has_int_m  <- EDA 12A candidate
  C4  customer belongs to the selected cohort (static attribute, e.g. loyalty_tier = 'gold')
  C1-C3 = "core" rows. CSAT / NPS availability is NOT required (would bias toward survey responders).

Selection
  * Scorecard maths shared with EDA Step 12 (cohort_scoring.score_candidates).
  * risk_signal uses complaints / open issues only - no sentiment ground truth in selection.
  * config/cohort_selection.json (all keys optional): {"core_rule": "any_two", "dimension": "loyalty_tier",
    "segment": "gold"}. Without dimension/segment the top-ranked candidate that passes the sample gate is used.

Run:  python step3d_build_cohort.py      (or: python run_pipeline.py --only 3d)

Change log (see decisions.md): D07 core rule aligned with the EDA (same-month sources + active month, configurable
CORE_RULE) instead of the undocumented 3-month lookback, D08 graceful failure (no traceback) + env-tunable gate,
D10 rule-size comparison report, D09 notebook-safe finish().
"""

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import json
from datetime import datetime

import pandas as pd

import business_rules as br
from cohort_scoring import max_mom_swing, score_candidates
from common import (CONFIG_DIR, COHORT_DIR, FEATURE_DIR, LABEL_COLUMNS, REPORT_DIR, SILVER_DIR,
                    checks_from, copy_to_csv, copy_to_parquet, finish, get_connection, log, print_checks, scalar)

OUT = REPORT_DIR / "cohort"
FEATURE_FILE = FEATURE_DIR / "customer_month_features.parquet"
LABEL_FILE = FEATURE_DIR / "customer_month_labels.parquet"
SELECTION_OVERRIDE = CONFIG_DIR / "cohort_selection.json"
COHORT_FILE = COHORT_DIR / "poc_customer_month.parquet"
COHORT_LABEL_FILE = COHORT_DIR / "poc_customer_month_labels.parquet"
COHORT_CUSTOMERS_FILE = COHORT_DIR / "poc_customers.parquet"

RISK_COND = "(complaint_count_m > 0 OR open_issue_count_m > 0)"       # no label-derived signal
SURVEY_COND = "(csat_n_m > 0 OR nps_n_m > 0)"
REPRESENTATIVENESS_COLS = ["purchase_amount_3m", "spend_pctile_m", "eng_event_count_3m", "active_days_3m",
                           "complaint_share_3m", "open_issue_share_3m", "failed_payment_rate_3m",
                           "sources_present_m", "months_observed", "ctx_tenure_months"]
SMD_ALERT = 0.25          # |standardised mean difference| above this = cohort differs materially


def cohort_conditions(rule: str) -> tuple[str, str]:
    """(HISTORY_COND, CORE_COND) on the Step 3c feature table - same definitions as the EDA."""
    history = br.history_sql("active_m")
    return history, f"{history} AND {br.core_source_sql('has_txn_m', 'has_eng_m', 'has_int_m', rule)}"


def load_override() -> dict:
    if not SELECTION_OVERRIDE.exists():
        return {}
    sel = json.loads(SELECTION_OVERRIDE.read_text())
    if ("dimension" in sel) != ("segment" in sel):
        raise ValueError("cohort_selection.json: give both 'dimension' and 'segment', or neither")
    if sel.get("core_rule", br.CORE_RULE) not in br.CORE_RULE_DESC:
        raise ValueError(f"cohort_selection.json: core_rule must be one of {list(br.CORE_RULE_DESC)}")
    return sel

## 2. Input views

In [ ]:
def register_inputs(con) -> None:
    for f in (FEATURE_FILE, LABEL_FILE, SILVER_DIR / "customers.parquet"):
        if not f.exists():
            raise FileNotFoundError(f"{f} not found - run Steps 3b and 3c first")
    con.execute(f"CREATE OR REPLACE VIEW feat AS SELECT * FROM read_parquet('{FEATURE_FILE.as_posix()}')")
    con.execute(f"CREATE OR REPLACE VIEW lbl AS SELECT * FROM read_parquet('{LABEL_FILE.as_posix()}')")
    # Static cohort attributes: one value per customer for the whole window, so membership never
    # changes month to month (ctx_tenure_months does change, so tenure_band uses the snapshot field).
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE cust_attr AS
        SELECT customer_id, customer_segment, loyalty_tier, region, preferred_channel,
               {br.tenure_band_sql('tenure_months')} AS tenure_band
        FROM read_parquet('{(SILVER_DIR / 'customers.parquet').as_posix()}')""")
    con.execute("""CREATE OR REPLACE TEMP TABLE fc AS
                   SELECT f.*, a.customer_segment, a.loyalty_tier, a.region, a.preferred_channel, a.tenure_band
                   FROM feat f JOIN cust_attr a USING (customer_id)""")

## 3. Core size under every rule (same comparison as EDA Step 12A)

In [ ]:
def rule_sizes(con) -> pd.DataFrame:
    rows = []
    for rule, desc in br.CORE_RULE_DESC.items():
        _, core = cohort_conditions(rule)
        n_cm, n_c = con.execute(f"SELECT count(*), count(DISTINCT customer_id) FROM fc WHERE {core}").fetchall()[0]
        rows.append({"core_rule": rule, "definition": desc, "core_customer_months": n_cm, "core_customers": n_c})
    return pd.DataFrame(rows)

## 4. Scorecard (same maths as EDA Step 12)

In [ ]:
def build_scorecard(con, history: str, core: str) -> pd.DataFrame:
    frames = []
    dims = [("ALL", "'ALL'")] + [(d, f"CAST({d} AS VARCHAR)") for d in br.COHORT_DIMS]
    for dim, expr in dims:
        d = con.sql(f"""
            SELECT '{dim}' AS dimension, COALESCE({expr}, 'unknown') AS segment,
                   count(DISTINCT customer_id) FILTER (WHERE {core})                       AS n_core_customers,
                   count(*) FILTER (WHERE {core})                                          AS n_core_customer_months,
                   round(100.0 * count(*) FILTER (WHERE {core})
                         / nullif(count(*) FILTER (WHERE {history}), 0), 2)                AS coverage_pct,
                   round(avg(active_months_to_date) FILTER (WHERE {core}), 2)              AS avg_active_months_history,
                   round(100.0 * count(*) FILTER (WHERE {core} AND {SURVEY_COND})
                         / nullif(count(*) FILTER (WHERE {core}), 0), 2)                   AS survey_label_pct,
                   round(100.0 * count(*) FILTER (WHERE {core} AND has_int_m = 1)
                         / nullif(count(*) FILTER (WHERE {core}), 0), 2)                   AS interaction_label_pct,
                   round(avg(spend_pctile_m) FILTER (WHERE {core}), 4)                    AS avg_spend_pctile,
                   round(stddev(spend_pctile_m) FILTER (WHERE {core}), 4)                 AS std_spend_pctile,
                   round(100.0 * count(*) FILTER (WHERE {core} AND {RISK_COND})
                         / nullif(count(*) FILTER (WHERE {core}), 0), 2)                   AS risk_signal_pct
            FROM fc GROUP BY ALL""").df()
        monthly = con.sql(f"""SELECT COALESCE({expr}, 'unknown') AS segment, month,
                                     count(*) FILTER (WHERE {core}) AS n
                              FROM fc WHERE months_observed >= {br.MIN_HISTORY_MONTHS} GROUP BY ALL""").df()
        frames.append(d.merge(max_mom_swing(monthly), on="segment", how="left"))
    return pd.concat(frames, ignore_index=True)


def choose_cohort(scored: pd.DataFrame, override: dict) -> dict:
    if "dimension" in override:
        row = scored[(scored.dimension == override["dimension"]) & (scored.segment == str(override["segment"]))]
        if row.empty:
            raise ValueError(f"Override {override} does not match any candidate in the scorecard")
        r = row.iloc[0]
        source = "business override (config/cohort_selection.json)"
    else:
        passing = scored[scored.passes_sample_gate]
        if passing.empty:
            raise ValueError(f"No candidate reaches {br.MIN_COHORT_CUSTOMERS:,} core customers - lower "
                             "MIN_COHORT_CUSTOMERS (env var) or pin a cohort in config/cohort_selection.json")
        r = passing.iloc[0]
        source = "top-ranked scorecard candidate"
    if r.dimension not in br.COHORT_DIMS:
        raise ValueError(f"Unsupported cohort dimension: {r.dimension}")
    return {"dimension": r.dimension, "segment": str(r.segment), "cohort_score": float(r.cohort_score),
            "n_core_customers": int(r.n_core_customers), "passes_sample_gate": bool(r.passes_sample_gate),
            "selection_source": source}

## 5. Cohort dataset + customer-grouped train/test split

In [ ]:
def build_cohort(con, chosen: dict, core: str) -> None:
    seg = chosen["segment"].replace("'", "''")
    member = f"COALESCE(CAST({chosen['dimension']} AS VARCHAR), 'unknown') = '{seg}'"
    test_cut = int(br.TEST_SHARE * 10_000)
    # Customer-level split from a deterministic hash: every month of a customer lands in ONE split.
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE poc_customers AS
        SELECT customer_id, '{chosen['dimension']}' AS cohort_dimension, '{seg}' AS cohort_segment,
               CASE WHEN hash(customer_id || '{br.SPLIT_SEED}') % 10000 < {test_cut}
                    THEN 'test' ELSE 'train' END AS split
        FROM cust_attr WHERE {member}
          AND customer_id IN (SELECT customer_id FROM fc WHERE {core})""")
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE poc AS
        SELECT f.* EXCLUDE (customer_segment, loyalty_tier, region, preferred_channel, tenure_band),
               p.cohort_dimension, p.cohort_segment, p.split
        FROM fc f JOIN poc_customers p USING (customer_id)
        WHERE {core}""")

## 6. Validation gate (cohort files are written only if every check passes)

In [ ]:
def validate(con, chosen: dict, rule: str) -> list[dict]:
    cols = [r[0] for r in con.execute("DESCRIBE poc").fetchall()]
    leak = [c for c in cols if any(lbl in c for lbl in LABEL_COLUMNS + ["interaction_text", "gt_"])]
    seg = chosen["segment"].replace("'", "''")
    src = br.core_source_sql("has_txn_m", "has_eng_m", "has_int_m", rule)
    return checks_from([
        ("unique customers >= MIN_COHORT_CUSTOMERS",
         max(0, br.MIN_COHORT_CUSTOMERS - scalar(con, "SELECT count(DISTINCT customer_id) FROM poc"))),
        ("duplicate customer-month", scalar(con, "SELECT count(*) - count(DISTINCT (customer_id, month)) FROM poc")),
        ("rows failing C2 active month", scalar(con, "SELECT count(*) FROM poc WHERE active_m <> 1")),
        ("rows failing C2 history", scalar(con, f"SELECT count(*) FROM poc WHERE months_observed < {br.MIN_HISTORY_MONTHS}")),
        (f"rows failing C3 sources ({rule})", scalar(con, f"SELECT count(*) FROM poc WHERE NOT {src}")),
        ("rows outside selected cohort", scalar(con, f"SELECT count(*) FROM poc WHERE cohort_segment <> '{seg}'")),
        ("label / text columns in model table", len(leak)),
        ("customers in both train and test",
         scalar(con, "SELECT count(*) FROM (SELECT customer_id FROM poc GROUP BY 1 HAVING count(DISTINCT split) > 1)")),
        ("test split empty", int(scalar(con, "SELECT count(*) FROM poc WHERE split = 'test'") == 0)),
        ("train split empty", int(scalar(con, "SELECT count(*) FROM poc WHERE split = 'train'") == 0)),
        ("cohort rows missing from feature table",
         scalar(con, "SELECT count(*) FROM poc p ANTI JOIN feat f USING (customer_id, month)")),
    ])


def representativeness(con, core: str) -> pd.DataFrame:
    """Cohort vs full core population (all segments). Informational - flags, does not fail."""
    rows = []
    for c in REPRESENTATIVENESS_COLS:
        r = con.execute(f"""
            SELECT (SELECT avg({c}) FROM poc), (SELECT avg({c}) FROM fc WHERE {core}),
                   (SELECT stddev({c}) FROM fc WHERE {core})""").fetchall()[0]
        smd = (r[0] - r[1]) / r[2] if None not in r and r[2] else None
        rows.append({"feature": c, "cohort_mean": r[0], "population_mean": r[1], "population_std": r[2],
                     "std_mean_diff": None if smd is None else round(smd, 3),
                     "flag": smd is not None and abs(smd) > SMD_ALERT})
    return pd.DataFrame(rows)


COVERAGE_SQL = """
    SELECT month, split, count(*) AS customer_months, count(DISTINCT customer_id) AS customers,
           round(100.0 * avg(has_txn_m), 1) AS pct_txn_m, round(100.0 * avg(has_eng_m), 1) AS pct_eng_m,
           round(100.0 * avg(has_int_m), 1) AS pct_int_m, round(avg(sources_present_m), 2) AS avg_sources_m,
           round(100.0 * avg(has_csat_3m::INT), 1) AS pct_csat_3m, round(100.0 * avg(has_nps_3m::INT), 1) AS pct_nps_3m
    FROM poc GROUP BY ALL ORDER BY month, split"""

## 7. Main

In [ ]:
def main() -> bool:
    for d in (COHORT_DIR, OUT):
        d.mkdir(parents=True, exist_ok=True)
    for f in (COHORT_FILE, COHORT_LABEL_FILE, COHORT_CUSTOMERS_FILE):
        f.unlink(missing_ok=True)      # never leave a stale cohort from an earlier run behind

    override = load_override()
    rule = override.get("core_rule", br.CORE_RULE)
    history, core = cohort_conditions(rule)
    con = get_connection()
    try:
        register_inputs(con)
        sizes = rule_sizes(con)
        sizes.to_csv(OUT / "cohort_rule_sizes.csv", index=False)
        print(sizes.to_string(index=False))

        log(f"Scoring candidate cohorts on the full core population (CORE_RULE = {rule})")
        sc = build_scorecard(con, history, core)
        scored = score_candidates(sc, br.MIN_COHORT_CUSTOMERS, br.COHORT_WEIGHTS)
        sc[sc.dimension == "ALL"].to_csv(OUT / "cohort_core_overall.csv", index=False)
        scored.to_csv(OUT / "cohort_scorecard.csv", index=False)
        print(scored.head(10)[["dimension", "segment", "n_core_customers", "coverage_pct",
                               "cohort_score", "passes_sample_gate"]].to_string(index=False))
        try:
            chosen = choose_cohort(scored, override)
        except ValueError as e:                 # expected business failure -> clean FAILED status, no traceback
            log(f"COHORT SELECTION FAILED: {e}")
            print("=" * 80 + "\nSTEP 3d STATUS: FAILED - cohort files NOT written")
            return False
        log(f"Selected cohort: {chosen['dimension']} = {chosen['segment']} ({chosen['selection_source']})")

        build_cohort(con, chosen, core)
        checks = validate(con, chosen, rule)
        rep = representativeness(con, core)
        rep.to_csv(OUT / "cohort_representativeness.csv", index=False)
        copy_to_csv(con, COVERAGE_SQL, OUT / "cohort_coverage_by_month.csv")
        pd.DataFrame(checks).to_csv(OUT / "cohort_validation.csv", index=False)
        ok = all(c["status"] == "PASS" for c in checks)
        if ok:   # write datasets only when the gate passes
            copy_to_parquet(con, "SELECT * FROM poc", COHORT_FILE)
            copy_to_parquet(con, """SELECT p.customer_id, p.month, p.split, l.* EXCLUDE (customer_id, month)
                                    FROM poc p JOIN lbl l USING (customer_id, month)""", COHORT_LABEL_FILE)
            copy_to_parquet(con, "SELECT * FROM poc_customers", COHORT_CUSTOMERS_FILE)
        stats = con.execute("""SELECT count(*), count(DISTINCT customer_id),
                                      count(DISTINCT customer_id) FILTER (WHERE split = 'train'),
                                      count(DISTINCT customer_id) FILTER (WHERE split = 'test') FROM poc""").fetchall()[0]
    finally:
        con.close()

    print("=" * 80)
    print(f"POC rows: {stats[0]:,} | customers: {stats[1]:,} (train {stats[2]:,} / test {stats[3]:,})")
    print_checks(checks)
    flagged = rep[rep["flag"]]["feature"].tolist()
    if flagged:
        print(f"  NOTE: cohort differs materially from the core population on {flagged} - "
              "report POC results as cohort performance, not full-population performance.")
    print("=" * 80 + f"\nSTEP 3d STATUS: {'PASSED' if ok else 'FAILED - cohort files NOT written'}")

    (OUT / "cohort_selection_used.json").write_text(json.dumps(
        {"run_at": datetime.now().isoformat(timespec="seconds"), **chosen,
         "criteria": {"core_rule": rule, "core_rule_definition": br.CORE_RULE_DESC[rule], "history": history,
                      "core": core, "risk_signal": RISK_COND, "min_cohort_customers": br.MIN_COHORT_CUSTOMERS,
                      "test_share": br.TEST_SHARE, "split_seed": br.SPLIT_SEED},
         "rows": stats[0], "customers": stats[1], "train_customers": stats[2], "test_customers": stats[3],
         "checks": checks, "representativeness_flags": flagged,
         "status": "PASSED" if ok else "FAILED"}, indent=2))
    return ok

In [ ]:
if __name__ == "__main__":
    finish(main())